# FMA Data Preprocessing

This notebook cleans the FMA metadata, emits per-subset CSVs, and converts MP3s into log-mel spectrograms saved to disk.

Two cleaned subsets are produced (controlled by `DATASET_SIZE` in `.env`):

- **small** — `subset == "small"` (≈ 8 000 tracks, 8 genres). Requires `fma_small/`. Audio existence is verified for every track.
- **medium** — `subset ∈ {small, medium}` (≈ 25 000 tracks, 16 genres). Requires `fma_small/` + `fma_metadata/`. If `fma_medium/` is present, medium spectrogram preprocessing is enabled; otherwise medium rows are kept in the CSV for Random Forest tabular features only and medium spectrogram preprocessing is skipped.

Spectrogram formats extracted (controlled by `PREPROCESS_FOR` in `.env`):

| Format | Clip | Shape | Directory |
|--------|------|-------|-----------|
| CNN    | 10 s | (1, 64, 1001) | `spectrograms_10/` |
| CRNN   | 20 s | (1, 64, 2001) | `spectrograms_20/` |

Run `DOWNLOAD_MEDIUM=1 bash setup.sh` to download `fma_medium/` (~22 GB).

## 1. Setup

Dependencies: `torch`, `torchaudio`, `pandas`, `numpy`, `tqdm`, `soundfile`.

Configuration is read from a `.env` file in the project root (or the parent of the current
working directory). Copy `.env.example` to `.env` and set the following variables:

| Variable | Default | Description |
|---|---|---|
| `DATASET_DIR` | *(auto-detected)* | Path to the directory containing `fma_small/`, `fma_medium/` and `fma_metadata/` |
| `PREPROCESSED_DIR` | `DATASET_DIR/fma_preprocessed` | Where cleaned CSVs and spectrograms are written |
| `DATASET_SIZE` | `small` | Which subset to process: `small`, `medium`, or `both` |

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio
import os

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable


def _load_dotenv(filename=".env"):
    """Load key=value pairs from a .env file without requiring python-dotenv."""
    for base in [Path.cwd(), Path.cwd().parent]:
        p = base / filename
        if p.exists():
            with open(p) as f:
                for line in f:
                    line = line.strip()
                    if line and not line.startswith("#") and "=" in line:
                        key, _, value = line.partition("=")
                        key = key.strip()
                        value = value.strip().strip('"').strip("'")
                        os.environ[key] = value
            print(f"Loaded {p}")
            return
    print("No .env file found — using environment variables / built-in defaults.")

_load_dotenv()

# ── Dataset size ──────────────────────────────────────────────────────────────
_size_raw = os.environ.get("DATASET_SIZE", "small").lower().strip()
RUN_SMALL  = _size_raw in ("small", "both")
RUN_MEDIUM = _size_raw in ("medium", "both")
if not RUN_SMALL and not RUN_MEDIUM:
    raise ValueError(f"DATASET_SIZE must be 'small', 'medium', or 'both'. Got: {_size_raw!r}")

# ── Spectrogram formats ──────────────────────────────────────────────────────
SPECTROGRAM_FORMATS = {
    "cnn":  {"clip_seconds": 10, "shape": (1, 64, 1001), "suffix": "10"},
    "crnn": {"clip_seconds": 20, "shape": (1, 64, 2001), "suffix": "20"},
}

_preprocess_raw = os.environ.get("PREPROCESS_FOR", "both").lower().strip()
if _preprocess_raw in ("", "none"):
    SELECTED_SPECTROGRAM_FORMATS = []
elif _preprocess_raw == "both":
    SELECTED_SPECTROGRAM_FORMATS = ["cnn", "crnn"]
else:
    SELECTED_SPECTROGRAM_FORMATS = [item.strip() for item in _preprocess_raw.split(",") if item.strip()]

_unknown_formats = sorted(set(SELECTED_SPECTROGRAM_FORMATS) - set(SPECTROGRAM_FORMATS))
if _unknown_formats:
    raise ValueError(
        "PREPROCESS_FOR must be 'cnn', 'crnn', 'both', 'none', or a comma-separated "
        f"list of cnn/crnn. Unknown: {_unknown_formats}"
    )

RUN_CNN = "cnn" in SELECTED_SPECTROGRAM_FORMATS
RUN_CRNN = "crnn" in SELECTED_SPECTROGRAM_FORMATS


# ── Paths ─────────────────────────────────────────────────────────────────────
if "DATASET_DIR" not in os.environ:
    raise EnvironmentError("DATASET_DIR is not set. Add it to your .env file.")

PROJECT_DIR   = Path(os.environ["DATASET_DIR"]).resolve()
metadata_path = PROJECT_DIR / "fma_metadata" / "tracks.csv"
if not metadata_path.exists():
    raise FileNotFoundError(
        f"fma_metadata/tracks.csv not found under DATASET_DIR={PROJECT_DIR}. "
        "Check your DATASET_DIR in .env."
    )
METADATA_PATH = metadata_path

_small_dir  = PROJECT_DIR / "fma_small"
_medium_dir = PROJECT_DIR / "fma_medium"
SMALL_AUDIO_DIR  = _small_dir.resolve()  if _small_dir.exists()  else None
MEDIUM_AUDIO_DIR = _medium_dir.resolve() if _medium_dir.exists() else None

if SMALL_AUDIO_DIR is None and MEDIUM_AUDIO_DIR is None:
    raise FileNotFoundError(
        f"Neither fma_small/ nor fma_medium/ found under DATASET_DIR={PROJECT_DIR}. "
        "Download at least one FMA subset."
    )

# fma_medium is a superset of fma_small — use it as fallback if fma_small is absent
if RUN_SMALL and SMALL_AUDIO_DIR is None:
    SMALL_AUDIO_DIR = MEDIUM_AUDIO_DIR
    print("Note: fma_small/ absent; reading small-subset tracks from fma_medium/.")

if RUN_MEDIUM and MEDIUM_AUDIO_DIR is None:
    print(
        "fma_medium not found. Medium CSVs will still be built; "
        "medium spectrogram preprocessing will be skipped."
    )

PROCESSED_DIR = (
    Path(os.environ["PREPROCESSED_DIR"]).resolve()
    if "PREPROCESSED_DIR" in os.environ
    else PROJECT_DIR / "fma_preprocessed"
)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"DATASET_DIR      : {PROJECT_DIR}")
print(f"SMALL_AUDIO_DIR  : {SMALL_AUDIO_DIR or '(not used)'}")
print(f"MEDIUM_AUDIO_DIR : {MEDIUM_AUDIO_DIR or '(not found)'}")
print(f"METADATA_PATH    : {METADATA_PATH}")
print(f"PREPROCESSED_DIR : {PROCESSED_DIR}")
print(f"device           : {device}")
print(f"RUN_SMALL        : {RUN_SMALL}  (DATASET_SIZE={_size_raw!r})")
print(f"RUN_MEDIUM       : {RUN_MEDIUM}")
print(f"PREPROCESS_FOR   : {_preprocess_raw!r}")
if SELECTED_SPECTROGRAM_FORMATS:
    print("SPECTROGRAM_FMT  :")
    for _format_name in SELECTED_SPECTROGRAM_FORMATS:
        _fmt = SPECTROGRAM_FORMATS[_format_name]
        print(
            f"  {_format_name.upper():4s}: {_fmt['clip_seconds']} s clips, "
            f"shape={_fmt['shape']}, dir=spectrograms_<size>_{_fmt['suffix']}/"
        )
else:
    print("SPECTROGRAM_FMT  : none")

Loaded /home/lily/acml-project/.env
DATASET_DIR      : /home/lily/acml-project
SMALL_AUDIO_DIR  : /home/lily/acml-project/fma_small
MEDIUM_AUDIO_DIR : /home/lily/acml-project/fma_medium
METADATA_PATH    : /home/lily/acml-project/fma_metadata/tracks.csv
PREPROCESSED_DIR : /home/lily/acml-project/fma_preprocessed
device           : cuda
RUN_SMALL        : True  (DATASET_SIZE='both')
RUN_MEDIUM       : True
PREPROCESS_FOR   : 'both'
SPECTROGRAM_FMT  :
  CNN : 10 s clips, shape=(1, 64, 1001), dir=spectrograms_<size>_10/
  CRNN: 20 s clips, shape=(1, 64, 2001), dir=spectrograms_<size>_20/


## 2. Load `tracks.csv`

`tracks.csv` uses a two-row header, so columns load as a `MultiIndex` like `("track", "genre_top")`. This stage selects the required fields and flattens them into a normal DataFrame.

In [2]:
def track_id_to_audio_path(track_id: int, subset: str = "small") -> Path:
    """Return the expected MP3 path for a track.

    Tracks with subset=='small' live in fma_small/.
    Tracks with subset=='medium' live in fma_medium/. If fma_medium/ is absent,
    keep the expected fma_medium/ path so medium spectrogram preprocessing can
    be skipped instead of falling back to fma_small/.
    """
    filename = f"{int(track_id):06d}.mp3"
    rel = Path(filename[:3]) / filename
    if subset == "medium":
        return (MEDIUM_AUDIO_DIR if MEDIUM_AUDIO_DIR is not None else _medium_dir) / rel
    return SMALL_AUDIO_DIR / rel


tracks = pd.read_csv(METADATA_PATH, index_col=0, header=[0, 1])

# The track_id index is reset so metadata_all stores it only as a column.
tracks_flat = tracks.reset_index(drop=True)

metadata_all = pd.DataFrame({
    "track_id": tracks.index.astype(int).to_numpy(),
    "subset":   tracks_flat[("set", "subset")].to_numpy(),
    "split":    tracks_flat[("set", "split")].to_numpy(),
    "genre":    tracks_flat[("track", "genre_top")].to_numpy(),
    "duration": tracks_flat[("track", "duration")].to_numpy(),
    "bit_rate": tracks_flat[("track", "bit_rate")].to_numpy(),
    "title":    tracks_flat[("track", "title")].to_numpy(),
    "artist":   tracks_flat[("artist", "name")].to_numpy(),
})
metadata_all["audio_path"] = metadata_all.apply(
    lambda row: track_id_to_audio_path(row["track_id"], row["subset"]), axis=1
)

print(f"Total tracks in tracks.csv: {len(metadata_all):,}")
print(metadata_all["subset"].value_counts())
metadata_all.head()

Total tracks in tracks.csv: 106,574
subset
large     81574
medium    17000
small      8000
Name: count, dtype: int64


,track_id,subset,split,genre,duration,bit_rate,title,artist,audio_path
0,2,small,training,Hip-Hop,168,256000,Food,AWOL,/home/lily/acml-project/fma_small/000/000002.mp3
1,3,medium,training,Hip-Hop,237,256000,Electric Ave,AWOL,/home/lily/acml-project/fma_medium/000/000003.mp3
2,5,small,training,Hip-Hop,206,256000,This World,AWOL,/home/lily/acml-project/fma_small/000/000005.mp3
3,10,small,training,Pop,161,192000,Freeway,Kurt Vile,/home/lily/acml-project/fma_small/000/000010.mp3
4,20,large,training,NaN,311,256000,Spiritual Level,Nicky Cook,/home/lily/acml-project/fma_small/000/000020.mp3


## 3. Clean each subset

Each subset drops tracks with a null genre, tracks in the `FAILED` list (known broken audio),
and tracks whose MP3 is missing or empty on disk.

Which subsets are cleaned depends on `DATASET_SIZE` in `.env`:
- **small** — `subset == "small"` (≈ 8 000 tracks, 8 genres). Audio-file existence is verified for every track.
- **medium** — `subset ∈ {small, medium}` (≈ 25 000 tracks, 16 genres). Audio checks only run for the small portion; medium-only rows are kept for Random Forest tabular features even without local audio files.

In [3]:
def keep(mask, df, label):
    old = len(df)
    df = df[mask]
    print(f"{label:42s} {old - len(df):>5d} dropped, {len(df):>5d} left")
    return df


def is_file_nonempty(path: Path) -> bool:
    try:
        return path.exists() and path.stat().st_size > 0
    except OSError:
        return False


FAILED = [
    1440, 26436, 28106, 29166, 29167, 29168, 29169, 29170, 29171, 29172,
    29173, 29179, 38903, 43903, 56757, 57603, 59361, 62095, 62954, 62956,
    62957, 62959, 62971, 75461, 80015, 86079, 92345, 92346, 92347, 92348,
    92349, 92350, 92351, 92352, 92353, 92354, 92355, 92356, 92357, 92358,
    92359, 92360, 92361, 96426, 104623, 106719, 109714, 114448, 114501, 114528,
    115235, 117759, 118003, 118004, 127827, 130296, 130298, 131076, 135804, 136486,
    144769, 144770, 144771, 144773, 144774, 144775, 144776, 144777, 144778, 152204,
    154923,
]

metadata = None
genre_to_idx_small = None

if RUN_SMALL:
    print("=== Cleaning small ===")
    metadata = metadata_all.copy()
    metadata = keep(metadata["subset"] == "small", metadata, "subset == small")
    metadata = keep(metadata["genre"].notna(), metadata, "genre not null")
    metadata = keep(metadata["audio_path"].map(is_file_nonempty), metadata, "audio file present and non-empty")
    metadata = keep(~metadata["track_id"].isin(FAILED), metadata, "not in FAILED list")

    metadata = metadata.sort_values("track_id").reset_index(drop=True)
    genres_small_sorted = sorted(metadata["genre"].unique())
    genre_to_idx_small = {g: i for i, g in enumerate(genres_small_sorted)}
    metadata["label"] = metadata["genre"].map(genre_to_idx_small).astype(int)

    print("\nSmall genre distribution:")
    print(metadata["genre"].value_counts().sort_index())
    print("\nSmall split distribution:")
    print(metadata["split"].value_counts())
else:
    print("Skipping small subset (DATASET_SIZE={!r}).".format(_size_raw))

=== Cleaning small ===
subset == small                            98574 dropped,  8000 left
genre not null                                 0 dropped,  8000 left
audio file present and non-empty               0 dropped,  8000 left
not in FAILED list                             0 dropped,  8000 left

Small genre distribution:
genre
Electronic       1000
Experimental     1000
Folk             1000
Hip-Hop          1000
Instrumental     1000
International    1000
Pop              1000
Rock             1000
Name: count, dtype: int64

Small split distribution:
split
training      6400
validation     800
test           800
Name: count, dtype: int64


In [4]:
metadata_medium = None
genre_to_idx_medium = None

if RUN_MEDIUM:
    print("=== Cleaning medium ===")
    metadata_medium = metadata_all.copy()

    metadata_medium = keep(
        metadata_medium["subset"].isin(["small", "medium"]),
        metadata_medium,
        "subset in {small, medium}",
    )
    metadata_medium = keep(metadata_medium["genre"].notna(), metadata_medium, "genre not null")
    metadata_medium = keep(~metadata_medium["track_id"].isin(FAILED), metadata_medium, "not in FAILED list")

    is_small_track = metadata_medium["subset"] == "small"
    has_audio      = metadata_medium["audio_path"].map(is_file_nonempty)

    if MEDIUM_AUDIO_DIR is not None:
        # fma_medium/ is present — require audio for all tracks.
        metadata_medium = keep(has_audio, metadata_medium, "audio file present and non-empty")
    else:
        # fma_medium/ absent — small tracks must have audio; medium-only tracks are
        # kept anyway so Random Forest can still use their tabular features.
        metadata_medium = keep(
            has_audio | (~is_small_track),
            metadata_medium,
            "small-portion audio present (medium-only kept for RF)",
        )

    metadata_medium = metadata_medium.sort_values("track_id").reset_index(drop=True)
    genres_medium_sorted = sorted(metadata_medium["genre"].unique())
    genre_to_idx_medium = {g: i for i, g in enumerate(genres_medium_sorted)}
    metadata_medium["label"] = metadata_medium["genre"].map(genre_to_idx_medium).astype(int)

    print("\nMedium genre distribution:")
    print(metadata_medium["genre"].value_counts().sort_index())
    print("\nMedium split distribution:")
    print(metadata_medium["split"].value_counts())
else:
    print("Skipping medium subset (DATASET_SIZE={!r}).".format(_size_raw))

=== Cleaning medium ===
subset in {small, medium}                  81574 dropped, 25000 left
genre not null                                 0 dropped, 25000 left
not in FAILED list                             0 dropped, 25000 left
audio file present and non-empty               0 dropped, 25000 left

Medium genre distribution:
genre
Blues                    74
Classical               619
Country                 178
Easy Listening           21
Electronic             6314
Experimental           2251
Folk                   1519
Hip-Hop                2201
Instrumental           1350
International          1018
Jazz                    384
Old-Time / Historic     510
Pop                    1186
Rock                   7103
Soul-RnB                154
Spoken                  118
Name: count, dtype: int64

Medium split distribution:
split
training      19922
test           2573
validation     2505
Name: count, dtype: int64


## 4. Save the cleaned CSVs

Each subset produces:
- `tracks_clean_{subset}.csv` — the full cleaned frame.
- `tracks_clean_{subset}_{split}.csv` — per-split slices for model training notebooks.
- `genre_to_idx_{subset}.csv` — the genre → label mapping.

In [5]:
def save_clean_csvs(frame: pd.DataFrame, subset_name: str, genre_to_idx: dict) -> None:
    base_path = PROCESSED_DIR / f"tracks_clean_{subset_name}.csv"
    frame.to_csv(base_path, index=False)
    print(f"Wrote {base_path} ({len(frame):,} rows)")

    for split_name in ("training", "validation", "test"):
        split_df = frame[frame["split"] == split_name]
        split_path = PROCESSED_DIR / f"tracks_clean_{subset_name}_{split_name}.csv"
        split_df.to_csv(split_path, index=False)
        print(f"Wrote {split_path} ({len(split_df):,} rows)")

    genre_map_path = PROCESSED_DIR / f"genre_to_idx_{subset_name}.csv"
    pd.DataFrame(
        sorted(genre_to_idx.items(), key=lambda kv: kv[1]),
        columns=["genre", "label"],
    ).to_csv(genre_map_path, index=False)
    print(f"Wrote {genre_map_path}")


if RUN_SMALL:
    save_clean_csvs(metadata, "small", genre_to_idx_small)

if RUN_MEDIUM:
    print()
    save_clean_csvs(metadata_medium, "medium", genre_to_idx_medium)

Wrote /home/lily/acml-project/fma_preprocessed/tracks_clean_small.csv (8,000 rows)
Wrote /home/lily/acml-project/fma_preprocessed/tracks_clean_small_training.csv (6,400 rows)
Wrote /home/lily/acml-project/fma_preprocessed/tracks_clean_small_validation.csv (800 rows)
Wrote /home/lily/acml-project/fma_preprocessed/tracks_clean_small_test.csv (800 rows)
Wrote /home/lily/acml-project/fma_preprocessed/genre_to_idx_small.csv

Wrote /home/lily/acml-project/fma_preprocessed/tracks_clean_medium.csv (25,000 rows)
Wrote /home/lily/acml-project/fma_preprocessed/tracks_clean_medium_training.csv (19,922 rows)
Wrote /home/lily/acml-project/fma_preprocessed/tracks_clean_medium_validation.csv (2,505 rows)
Wrote /home/lily/acml-project/fma_preprocessed/tracks_clean_medium_test.csv (2,573 rows)
Wrote /home/lily/acml-project/fma_preprocessed/genre_to_idx_medium.csv


## 5. Tabular feature extraction

Filters `fma_metadata/features.csv` to the cleaned track IDs for each subset and saves the results to `fma_preprocessed/`. MLP and XGBoost load these files via `tabular_data.py`.

In [6]:
features_path = PROJECT_DIR / "fma_metadata" / "features.csv"
features_all = None

if features_path.exists():
    features_all = pd.read_csv(features_path, index_col=0, header=[0, 1, 2])
    print(f"Loaded {features_path}  shape={features_all.shape}")
else:
    print(f"features.csv not found at {features_path}. Skipping tabular feature extraction.")

if features_all is not None:
    if RUN_SMALL:
        small_ids = set(metadata["track_id"])
        features_small = features_all[features_all.index.isin(small_ids)]
        out = PROCESSED_DIR / "features_small.csv"
        features_small.to_csv(out)
        print(f"Wrote {out}  ({len(features_small):,} rows, {features_small.shape[1]} features)")

    if RUN_MEDIUM:
        medium_ids = set(metadata_medium["track_id"])
        features_medium = features_all[features_all.index.isin(medium_ids)]
        out = PROCESSED_DIR / "features_medium.csv"
        features_medium.to_csv(out)
        print(f"Wrote {out}  ({len(features_medium):,} rows, {features_medium.shape[1]} features)")

Loaded /home/lily/acml-project/fma_metadata/features.csv  shape=(106574, 518)
Wrote /home/lily/acml-project/fma_preprocessed/features_small.csv  (8,000 rows, 518 features)
Wrote /home/lily/acml-project/fma_preprocessed/features_medium.csv  (25,000 rows, 518 features)


## 6. Spectrogram extraction

CNN and CRNN use different clip lengths, which produce different spectrogram shapes:

| Model | Clip length | Output shape | Output directory |
|-------|-------------|--------------|------------------|
| CNN   | 10 s        | (1, 64, 1001) | `spectrograms_<size>_10/` |
| CRNN  | 20 s        | (1, 64, 2001) | `spectrograms_<size>_20/` |

Common settings: 16 kHz mono, `n_fft=400`, `hop_length=160`, 64 mel bins, log power scale,
per-clip z-score normalisation, saved as `float32` `.npy` files.

Only formats selected by `PREPROCESS_FOR` are extracted for the selected `DATASET_SIZE`.

In [7]:
SAMPLE_RATE = 16_000
N_FFT       = 400
HOP_LENGTH  = 160
N_MELS      = 64


def _make_transforms(clip_seconds: int):
    num_samples = SAMPLE_RATE * clip_seconds
    mel = torchaudio.transforms.MelSpectrogram(
        sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS,
    ).to(device)
    to_db = torchaudio.transforms.AmplitudeToDB(stype="power").to(device)
    return mel, to_db, num_samples


def load_clip(path: Path, num_samples: int) -> torch.Tensor:
    audio, sr = sf.read(path, dtype="float32", always_2d=True)
    waveform = torch.from_numpy(audio.T).mean(dim=0, keepdim=True)
    if sr != SAMPLE_RATE:
        waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)
    if waveform.shape[1] > num_samples:
        start = (waveform.shape[1] - num_samples) // 2
        waveform = waveform[:, start:start + num_samples]
    elif waveform.shape[1] < num_samples:
        waveform = torch.nn.functional.pad(waveform, (0, num_samples - waveform.shape[1]))
    return torch.nan_to_num(waveform, nan=0.0, posinf=0.0, neginf=0.0).clamp(-1.0, 1.0)


@torch.no_grad()
def waveform_to_spec(waveform: torch.Tensor, mel_transform, to_db) -> torch.Tensor:
    spec = to_db(mel_transform(waveform.to(device)))
    spec = torch.nan_to_num(spec, nan=0.0, posinf=0.0, neginf=0.0)
    mean = spec.mean(dim=(-2, -1), keepdim=True)
    std  = spec.std(dim=(-2, -1), keepdim=True).clamp_min(1e-6)
    return torch.nan_to_num((spec - mean) / std, nan=0.0, posinf=0.0, neginf=0.0).cpu()


def extract_spectrograms(
    metadata_df: pd.DataFrame,
    clip_seconds: int,
    spec_dir: Path,
    manifest_prefix: str,
    skip_existing: bool = True,
) -> pd.DataFrame:
    """Extract mel spectrograms for every row in metadata_df and save manifests.

    Returns the manifest DataFrame (one row per successfully extracted spectrogram).
    """
    source_df = metadata_df[metadata_df["audio_path"].map(lambda path: is_file_nonempty(Path(path)))].copy()
    missing_audio = len(metadata_df) - len(source_df)
    if missing_audio:
        print(f"Skipping {missing_audio:,} row(s) without local audio for {manifest_prefix}.")

    mel, to_db, num_samples = _make_transforms(clip_seconds)
    spec_dir.mkdir(parents=True, exist_ok=True)

    def _spec_path(track_id: int) -> Path:
        fname = f"{int(track_id):06d}.npy"
        return spec_dir / fname[:3] / fname

    spec_paths, dropped_rows = [], []
    n_skipped = n_written = 0

    for row in tqdm(source_df.itertuples(index=False), total=len(source_df),
                    desc=f"Extracting {manifest_prefix}"):
        out_path = _spec_path(row.track_id)
        out_path.parent.mkdir(parents=True, exist_ok=True)

        if skip_existing and out_path.exists():
            spec_paths.append(str(out_path))
            n_skipped += 1
            continue

        try:
            waveform = load_clip(Path(row.audio_path), num_samples)
            spec = waveform_to_spec(waveform, mel, to_db).numpy().astype(np.float32)
        except Exception as exc:
            dropped_rows.append({
                "track_id": row.track_id,
                "audio_path": str(row.audio_path),
                "error": f"{type(exc).__name__}: {exc}",
            })
            spec_paths.append(None)
            continue

        np.save(out_path, spec)
        spec_paths.append(str(out_path))
        n_written += 1

    print(f"\nWrote {n_written:,} new, reused {n_skipped:,} existing, dropped {len(dropped_rows):,} on error.")
    if dropped_rows:
        print(pd.DataFrame(dropped_rows).to_string(index=False))

    manifest_df = source_df.copy()
    manifest_df["spectrogram_path"] = spec_paths
    manifest_df = manifest_df[manifest_df["spectrogram_path"].notna()].reset_index(drop=True)
    cols = ["track_id", "split", "genre", "label", "spectrogram_path",
            "audio_path", "duration", "bit_rate", "title", "artist"]
    manifest_df = manifest_df[cols]

    manifest_path = PROCESSED_DIR / f"{manifest_prefix}_manifest.csv"
    manifest_df.to_csv(manifest_path, index=False)
    print(f"Wrote {manifest_path} ({len(manifest_df):,} rows)")

    for split_name in ("training", "validation", "test"):
        split_df = manifest_df[manifest_df["split"] == split_name]
        split_path = PROCESSED_DIR / f"{manifest_prefix}_{split_name}.csv"
        split_df.to_csv(split_path, index=False)
        print(f"Wrote {split_path} ({len(split_df):,} rows)")

    if dropped_rows:
        err_path = PROCESSED_DIR / f"{manifest_prefix}_extraction_errors.csv"
        pd.DataFrame(dropped_rows).to_csv(err_path, index=False)
        print(f"Wrote {err_path} ({len(dropped_rows):,} rows)")

    return manifest_df


print("Spectrogram helpers defined.")

Spectrogram helpers defined.


## 7. CNN spectrogram extraction (10 s clips → `spectrograms_<size>_10/`)

Extracts from the subset(s) selected by `DATASET_SIZE`.

In [8]:
manifest_cnn_small  = None
manifest_cnn_medium = None

if RUN_CNN and RUN_SMALL:
    print("=== CNN · small (10 s clips) ===")
    manifest_cnn_small = extract_spectrograms(
        metadata,
        clip_seconds=10,
        spec_dir=PROCESSED_DIR / "spectrograms_small_10",
        manifest_prefix="spectrograms_small_10",
    )

if RUN_CNN and RUN_MEDIUM:
    print("\n=== CNN · medium (10 s clips) ===")
    if MEDIUM_AUDIO_DIR is None:
        print("fma_medium not found. Skipping medium spectrogram preprocessing.")
    else:
        manifest_cnn_medium = extract_spectrograms(
            metadata_medium,
            clip_seconds=10,
            spec_dir=PROCESSED_DIR / "spectrograms_medium_10",
            manifest_prefix="spectrograms_medium_10",
        )

=== CNN · small (10 s clips) ===


Extracting spectrograms_small_10:   0%|          | 0/8000 [00:00<?, ?it/s]

Note: Illegal Audio-MPEG-Header 0x00000000 at offset 33361.
Note: Trying to resync...
Note: Skipped 1024 bytes in input.
[src/libmpg123/parse.c:wetwork():1349] error: Giving up resync after 1024 bytes - your stream is not nice... (maybe increasing resync limit could help).
Note: Illegal Audio-MPEG-Header 0x00000000 at offset 22401.
Note: Trying to resync...
Note: Skipped 1024 bytes in input.
[src/libmpg123/parse.c:wetwork():1349] error: Giving up resync after 1024 bytes - your stream is not nice... (maybe increasing resync limit could help).
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
Note: Illegal Audio-MPEG-Header 0x00000000 at offset 63168.
Note: Trying to resync...
Note: Skipped 1024 bytes in input.
[src/libmpg123/parse.c:wetwork():1349] error: Giving up resync after 1024 bytes - your stream is not nice... (maybe increasing resync limit could help).
[src/libmpg123/parse.c:do_readahead():1083] warning: Cannot read next header, a one-frame stream? D


Wrote 0 new, reused 7,994 existing, dropped 6 on error.
 track_id                                       audio_path                                                                                                                                               error
    98565 /home/lily/acml-project/fma_small/098/098565.mp3                                                                                                        LibsndfileError: Unspecified internal error.
    98567 /home/lily/acml-project/fma_small/098/098567.mp3                                                                                                        LibsndfileError: Unspecified internal error.
    98569 /home/lily/acml-project/fma_small/098/098569.mp3                                                                                                        LibsndfileError: Unspecified internal error.
    99134 /home/lily/acml-project/fma_small/099/099134.mp3 LibsndfileError: Error opening '/home/lily/acml-project/

Extracting spectrograms_medium_10:   0%|          | 0/25000 [00:00<?, ?it/s]

[src/libmpg123/parse.c:do_readahead():1083] warning: Cannot read next header, a one-frame stream? Duh...
[src/libmpg123/parse.c:do_readahead():1083] warning: Cannot read next header, a one-frame stream? Duh...
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/parse.c:do_readahead():1083] warning: Cannot read next header, a one-frame stream? Duh...
[src/libmpg123/layer3.c:INT123_do_layer3():1844] error: dequantization failed!
[src/libmpg123/parse.c:do_readahead():1083] warning: Cannot read next header, a one-frame stream? Duh...
[src/libmpg123/parse.c:do_readahead():1083] warning: Cannot read next header, a one-frame stream? Duh...
[


Wrote 13,877 new, reused 11,103 existing, dropped 20 on error.
 track_id                                        audio_path                                                                                                                                                error
     1486 /home/lily/acml-project/fma_medium/001/001486.mp3 LibsndfileError: Error opening '/home/lily/acml-project/fma_medium/001/001486.mp3': File does not exist or is not a regular file (possibly a pipe?).
     5574 /home/lily/acml-project/fma_medium/005/005574.mp3 LibsndfileError: Error opening '/home/lily/acml-project/fma_medium/005/005574.mp3': File does not exist or is not a regular file (possibly a pipe?).
    65753 /home/lily/acml-project/fma_medium/065/065753.mp3 LibsndfileError: Error opening '/home/lily/acml-project/fma_medium/065/065753.mp3': File does not exist or is not a regular file (possibly a pipe?).
    80391 /home/lily/acml-project/fma_medium/080/080391.mp3 LibsndfileError: Error opening '/home/li

## 8. CRNN spectrogram extraction (20 s clips → `spectrograms_<size>_20/`)

Extracts from the subset(s) selected by `DATASET_SIZE`.

In [9]:
manifest_crnn_small  = None
manifest_crnn_medium = None

if RUN_CRNN and RUN_SMALL:
    print("=== CRNN · small (20 s clips) ===")
    manifest_crnn_small = extract_spectrograms(
        metadata,
        clip_seconds=20,
        spec_dir=PROCESSED_DIR / "spectrograms_small_20",
        manifest_prefix="spectrograms_small_20",
    )

if RUN_CRNN and RUN_MEDIUM:
    print("\n=== CRNN · medium (20 s clips) ===")
    if MEDIUM_AUDIO_DIR is None:
        print("fma_medium not found. Skipping medium spectrogram preprocessing.")
    else:
        manifest_crnn_medium = extract_spectrograms(
            metadata_medium,
            clip_seconds=20,
            spec_dir=PROCESSED_DIR / "spectrograms_medium_20",
            manifest_prefix="spectrograms_medium_20",
        )

=== CRNN · small (20 s clips) ===


Extracting spectrograms_small_20:   0%|          | 0/8000 [00:00<?, ?it/s]


Wrote 0 new, reused 7,994 existing, dropped 6 on error.
 track_id                                       audio_path                                                                                                                                               error
    98565 /home/lily/acml-project/fma_small/098/098565.mp3                                                                                                        LibsndfileError: Unspecified internal error.
    98567 /home/lily/acml-project/fma_small/098/098567.mp3                                                                                                        LibsndfileError: Unspecified internal error.
    98569 /home/lily/acml-project/fma_small/098/098569.mp3                                                                                                        LibsndfileError: Unspecified internal error.
    99134 /home/lily/acml-project/fma_small/099/099134.mp3 LibsndfileError: Error opening '/home/lily/acml-project/

Note: Illegal Audio-MPEG-Header 0x00000000 at offset 33361.
Note: Trying to resync...
Note: Skipped 1024 bytes in input.
[src/libmpg123/parse.c:wetwork():1349] error: Giving up resync after 1024 bytes - your stream is not nice... (maybe increasing resync limit could help).
Note: Illegal Audio-MPEG-Header 0x00000000 at offset 22401.
Note: Trying to resync...
Note: Skipped 1024 bytes in input.
[src/libmpg123/parse.c:wetwork():1349] error: Giving up resync after 1024 bytes - your stream is not nice... (maybe increasing resync limit could help).
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
Note: Illegal Audio-MPEG-Header 0x00000000 at offset 63168.
Note: Trying to resync...
Note: Skipped 1024 bytes in input.
[src/libmpg123/parse.c:wetwork():1349] error: Giving up resync after 1024 bytes - your stream is not nice... (maybe increasing resync limit could help).
[src/libmpg123/parse.c:do_readahead():1083] warning: Cannot read next header, a one-frame stream? D

Wrote /home/lily/acml-project/fma_preprocessed/spectrograms_small_20_training.csv (6,394 rows)
Wrote /home/lily/acml-project/fma_preprocessed/spectrograms_small_20_validation.csv (800 rows)
Wrote /home/lily/acml-project/fma_preprocessed/spectrograms_small_20_test.csv (800 rows)
Wrote /home/lily/acml-project/fma_preprocessed/spectrograms_small_20_extraction_errors.csv (6 rows)

=== CRNN · medium (20 s clips) ===


Extracting spectrograms_medium_20:   0%|          | 0/25000 [00:00<?, ?it/s]

[src/libmpg123/parse.c:do_readahead():1083] warning: Cannot read next header, a one-frame stream? Duh...
[src/libmpg123/parse.c:do_readahead():1083] warning: Cannot read next header, a one-frame stream? Duh...
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/parse.c:do_readahead():1083] warning: Cannot read next header, a one-frame stream? Duh...
[src/libmpg123/layer3.c:INT123_do_layer3():1844] error: dequantization failed!
[src/libmpg123/parse.c:do_readahead():1083] warning: Cannot read next header, a one-frame stream? Duh...
[src/libmpg123/parse.c:do_readahead():1083] warning: Cannot read next header, a one-frame stream? Duh...
[


Wrote 13,877 new, reused 11,103 existing, dropped 20 on error.
 track_id                                        audio_path                                                                                                                                                error
     1486 /home/lily/acml-project/fma_medium/001/001486.mp3 LibsndfileError: Error opening '/home/lily/acml-project/fma_medium/001/001486.mp3': File does not exist or is not a regular file (possibly a pipe?).
     5574 /home/lily/acml-project/fma_medium/005/005574.mp3 LibsndfileError: Error opening '/home/lily/acml-project/fma_medium/005/005574.mp3': File does not exist or is not a regular file (possibly a pipe?).
    65753 /home/lily/acml-project/fma_medium/065/065753.mp3 LibsndfileError: Error opening '/home/lily/acml-project/fma_medium/065/065753.mp3': File does not exist or is not a regular file (possibly a pipe?).
    80391 /home/lily/acml-project/fma_medium/080/080391.mp3 LibsndfileError: Error opening '/home/li

## 9. Sanity check

Loads the first spectrogram from each generated manifest and prints shape, dtype, and value statistics.

In [10]:
checks = [
    ("CNN   · small",  manifest_cnn_small),
    ("CNN   · medium", manifest_cnn_medium),
    ("CRNN  · small",  manifest_crnn_small),
    ("CRNN  · medium", manifest_crnn_medium),
]

any_checked = False
for label, mf in checks:
    if mf is None or len(mf) == 0:
        continue
    any_checked = True
    first = mf.iloc[0]
    loaded = np.load(first["spectrogram_path"])
    print(f"[{label}]  track {first['track_id']} ({first['genre']}, {first['split']})")
    print(f"  shape={loaded.shape}  dtype={loaded.dtype}  "
          f"mean={loaded.mean():.4f}  std={loaded.std():.4f}  "
          f"min={loaded.min():.2f}  max={loaded.max():.2f}")

if not any_checked:
    print("No spectrograms were extracted (PREPROCESS_FOR='none' or no subsets selected).")

[CNN   · small]  track 2 (Hip-Hop, training)
  shape=(1, 64, 1001)  dtype=float32  mean=-0.0000  std=1.0000  min=-6.00  max=3.76
[CNN   · medium]  track 2 (Hip-Hop, training)
  shape=(1, 64, 1001)  dtype=float32  mean=-0.0000  std=1.0000  min=-6.00  max=3.76
[CRNN  · small]  track 2 (Hip-Hop, training)
  shape=(1, 64, 2001)  dtype=float32  mean=-0.0000  std=1.0000  min=-5.90  max=3.85
[CRNN  · medium]  track 2 (Hip-Hop, training)
  shape=(1, 64, 2001)  dtype=float32  mean=-0.0000  std=1.0000  min=-5.90  max=3.85
